# 第 21 章　ランダムフォレスト

同じ配合と材齢でも，学習用の記録を引き直して作った二本の木は，違う値を予測しました。一本ずつの予測がかなり違うとき，多くの木を作って平均するとどうなるでしょうか。Book 第 21 章と一緒に，前章の架空の六行で葉の値を計算し，実データで個々の木の答えと森の平均を比べます。実習では，検証の一行を自分で選びます。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。ローカルでは配布 ZIP の配置で開くか，ノートと CSV を同じフォルダーに置きます。手元にファイルがなければ公開 URL から CSV を読み込みます。

## 講義　記録を引き直して予測を合わせる

### 六行を引き直す

前章の架空例と同じ六行 A〜F です。復元抽出で `A, A, B, D, E, F` を得た場合は，A が二回入って C は入りません。図の三段は，違う三組の行から作った深さ 1 の木です。丸は学習に使う記録で，重複したものは大きく描きます。縦の線は材齢 21 日です。各木の 21 日の予測をひし形から読み，三本の予測を平均してください。

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.tree import DecisionTreeRegressor

toy = pd.DataFrame({
    "age": [3, 7, 14, 28, 56, 90],
    "strength": [5, 10, 15, 30, 35, 40],
}, index=list("ABCDEF"))
draws = ["AABDEF", "ABCCEF", "ABDDEF"]
toy_input = pd.DataFrame({"age": [21]})
toy_predictions = []

fig, axes = plt.subplots(3, 1, sharex=True, sharey=True, figsize=(7.2, 5.5), layout="constrained")
for number, (ax, draw) in enumerate(zip(axes, draws), start=1):
    drawn = toy.loc[list(draw)]
    tree = DecisionTreeRegressor(max_depth=1, random_state=2026)
    tree.fit(drawn[["age"]], drawn["strength"])
    boundary = tree.tree_.threshold[0]
    leaves = tree.predict(pd.DataFrame({"age": [0, 95]}))
    prediction = tree.predict(toy_input)[0]
    toy_predictions.append(prediction)

    marks = drawn.groupby("age")["strength"].agg(["first", "size"])
    ax.step([0, boundary, 95], [leaves[0], leaves[1], leaves[1]],
            where="post", color="#286b70", linewidth=1.5)
    ax.scatter(marks.index, marks["first"], s=marks["size"] * 24, color="#286b70")
    ax.axvline(21, color="#687581", linestyle="--", linewidth=1)
    ax.scatter(21, prediction, marker="D", s=39, color="#a34f30", zorder=4)
    ax.set(ylabel=f"Tree {number}\nMPa", xlim=(0, 95), ylim=(0, 45))
axes[-1].set(xlabel="Age (days)", xticks=[0, 21, 35, 56, 90])
plt.show()

### コンクリートの記録を二回引く

材料の量と試験する材齢が指定された段階で，測る圧縮強度を MPa で予測します。七つの材料量が完全に同じ配合はまとめて区分したので，同じ配合の別の材齢が学習と検証に分かれることはありません。学習 631 行と検証 208 行を前章と同じように使い，保留の成績は計算しません。

学習用の 631 行から復元抽出し，深さ 2 の木を作ります。次の木では新たに行を引き直し，木の設定はそろえます。同じ検証の一行を二本に入れて，初めに見る特徴量と，最後に予測する値を確認します。

In [ ]:
from pathlib import Path

file_name = "concrete-strength.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

concrete = pd.read_csv(csv_path)
training = concrete.loc[concrete["split"] == "train"]
validation = concrete.loc[concrete["split"] == "validation"].set_index("record_id")
features = [
    "cement", "slag", "fly_ash", "water", "superplasticizer",
    "coarse_aggregate", "fine_aggregate", "age",
]

X_train = training[features]
y_train = training["strength"]
X_valid = validation[features]
y_valid = validation["strength"]
comparison_input = X_valid.loc[[1]]

In [ ]:
import numpy as np

rng = np.random.default_rng(2026)
resampled_roots = []
for number in [1, 2]:
    positions = rng.choice(len(X_train), size=len(X_train), replace=True)
    shallow = DecisionTreeRegressor(max_depth=2, random_state=2026)
    shallow.fit(X_train.iloc[positions], y_train.iloc[positions])
    resampled_roots.append({
        "tree": number,
        "distinct_rows": len(np.unique(positions)),
        "first_feature": features[shallow.tree_.feature[0]],
        "first_threshold": shallow.tree_.threshold[0],
        "prediction": shallow.predict(comparison_input)[0],
    })

pd.DataFrame(resampled_roots).round(2)

### 森を作る

学習行を木ごとに復元抽出し，分岐を探すときは，8 列のうち候補にする列を毎回選びます。今回は `max_features=0.7` なので，分岐での候補数は通常 5 列です。

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest = RandomForestRegressor(
    n_estimators=200,
    max_features=0.7,
    bootstrap=True,
    random_state=2026,
)
forest.fit(X_train, y_train)

### 候補の列だけを変える

`max_features=1.0` にした森を，同じ乱数設定で作ります。6 本目の復元抽出が一致するか，表の最初の条件で調べる列も一致するかを確かめます。学習に使う行を変えた最初の実演と，何を変えた点が違うでしょうか。

続けて，最初の設定の森で，引いた行と引かなかった行を比べます。重複を一つと数えると何行が入ったか，両方に現れる配合はいくつかを読みます。

In [ ]:
all_columns_forest = RandomForestRegressor(
    n_estimators=200, max_features=1.0, bootstrap=True, random_state=2026,
).fit(X_train, y_train)

tree_index = 5  # 0 から数えて 6 本目
same_draw = np.array_equal(
    forest.estimators_samples_[tree_index],
    all_columns_forest.estimators_samples_[tree_index],
)
print("引いた行が同じ:", same_draw)

root_comparison = []
for candidate_count, model in [(5, forest), (8, all_columns_forest)]:
    root = model.estimators_[tree_index].tree_
    root_comparison.append({
        "candidate_columns": candidate_count,
        "first_feature": features[root.feature[0]],
        "first_threshold": root.threshold[0],
    })
pd.DataFrame(root_comparison).round(2)

In [ ]:
first_draw = training.iloc[forest.estimators_samples_[0]]
unused_rows = training.loc[~training["record_id"].isin(first_draw["record_id"])]
shared_recipes = set(first_draw["recipe_id"]) & set(unused_rows["recipe_id"])

print(f"Rows in first tree's draw: {len(first_draw):,}")
print(f"Distinct original rows:    {first_draw['record_id'].nunique():,}")
print(f"Recipes on both sides:     {len(shared_recipes):,}")

### 一本ずつ予測して平均する

検証の `record_id=1` をすべての木に渡して予測します。最初の五つの値と 200 本の平均を見ます。`forest.predict()` の答えは，200 本の算術平均と一致しますか。最初の一本と同じ予測値を 200 個並べた場合の平均は，どうなるでしょうか。

In [ ]:
def predict_from_each_tree(forest, one_row):
    values = one_row.to_numpy()
    return np.array([tree.predict(values)[0] for tree in forest.estimators_])

individual = predict_from_each_tree(forest, comparison_input)
mean_prediction = individual.mean()
forest_prediction = forest.predict(comparison_input)[0]

print("Number of trees:", len(individual))
print("First five predictions (MPa):", individual[:5].round(2))
print(f"Average of trees:  {mean_prediction:.2f} MPa")
print(f"Forest prediction: {forest_prediction:.2f} MPa")
print("Same value:", np.isclose(mean_prediction, forest_prediction))

### 二行で外れ方を比べる

深さ制限なしの一本の木と，森の平均を実測に照らします。記録 1 と 402 の図で，実測の線より低く答えた木と，高く答えた木がどこにあるかを見ます。どちらの記録で平均した予測は実測に近づきましたか。

In [ ]:
single_tree = DecisionTreeRegressor(random_state=2026)
single_tree.fit(X_train, y_train)

evaluation = validation[["recipe_id", "cement", "age", "strength"]].copy()
evaluation["single_prediction"] = single_tree.predict(X_valid)
evaluation["forest_prediction"] = forest.predict(X_valid)
evaluation["single_abs_error"] = (evaluation["strength"] - evaluation["single_prediction"]).abs()
evaluation["forest_abs_error"] = (evaluation["strength"] - evaluation["forest_prediction"]).abs()

example_ids = [1, 402]
evaluation.loc[example_ids].round(2)

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, sharey=True, figsize=(7.2, 5.2), layout="constrained")
for ax, record in zip(axes, example_ids):
    predictions = predict_from_each_tree(forest, X_valid.loc[[record]])
    sampled_row = evaluation.loc[record]
    ax.hist(predictions, bins=np.arange(20, 86, 5), color="#9abcc0",
            edgecolor="white", linewidth=0.8)
    ax.axvline(sampled_row["strength"], color="#283845", linewidth=2, label="Observed")
    ax.axvline(sampled_row["single_prediction"], color="#596878", linewidth=1.6,
               linestyle="--", label="Single tree")
    ax.axvline(sampled_row["forest_prediction"], color="#a34f30", linewidth=2,
               linestyle="-.", label="Forest mean")
    ax.set(title=f"Record {record}", ylabel="Number of trees", xlim=(20, 85))
axes[0].legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.14),
               ncols=3, fontsize=9)
axes[-1].set_xlabel("Compressive strength (MPa)")
plt.show()

### 平均のばらつき

記録 1 の 200 本を 50 本ずつ四組に分けます。それぞれの組で，先頭の 5 本と全 50 本の平均を比べます。平均した値は，組が変わるとどのくらい違うでしょうか。四組の平均同士の違いと，実測の約 79.99 MPa からのずれを分けて考えます。

In [ ]:
groups = individual.reshape(4, 50)
means_of_five = groups[:, :5].mean(axis=1)
means_of_fifty = groups.mean(axis=1)
pd.DataFrame({
    "trees_per_group": [5, 50],
    "lowest_mean": [means_of_five.min(), means_of_fifty.min()],
    "highest_mean": [means_of_five.max(), means_of_fifty.max()],
}).round(2)

### 検証全体で比べる

学習と検証それぞれの MAE を比べます。深さ制限なしの一本と森で，学習ではどちらの MAE が小さく，検証ではどうなったでしょうか。誤差の散布図では，一点は検証で使った一行です。斜めの破線より下にある行では，森の方が実測に近くなります。二行だけを見たときと，全体をまとめたときの言い方を考えます。

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

linear = LinearRegression().fit(X_train, y_train)
shallow_tree = DecisionTreeRegressor(max_depth=2, random_state=2026).fit(X_train, y_train)
baseline = y_train.mean()
scores = [{
    "model": "Constant baseline",
    "training_mae": (y_train - baseline).abs().mean(),
    "validation_mae": (y_valid - baseline).abs().mean(),
}]
for label, model in [
    ("Linear regression", linear),
    ("Tree, depth 2", shallow_tree),
    ("Tree, unrestricted depth", single_tree),
    ("Forest, 200 trees", forest),
]:
    scores.append({
        "model": label,
        "training_mae": mean_absolute_error(y_train, model.predict(X_train)),
        "validation_mae": mean_absolute_error(y_valid, model.predict(X_valid)),
    })

pd.DataFrame(scores).set_index("model").round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.25), layout="constrained")
errors = evaluation[["single_abs_error", "forest_abs_error"]]
limit = errors.to_numpy().max() * 1.1
ax.plot([0, limit], [0, limit], color="#687581", linewidth=1, linestyle="--")
ax.scatter(errors["single_abs_error"], errors["forest_abs_error"],
           s=17, color="#286b70", alpha=0.46, linewidths=0)
for record in example_ids:
    x, y = errors.loc[record]
    ax.scatter(x, y, marker="D", s=43, color="#a34f30", zorder=3)
    ax.annotate(str(record), (x, y), xytext=(6, 5), textcoords="offset points")
ax.set(xlabel="Single-tree absolute error (MPa)", ylabel="Forest absolute error (MPa)",
       xlim=(0, limit), ylim=(0, limit), aspect="equal")
plt.show()

In [ ]:
gain = evaluation["single_abs_error"] - evaluation["forest_abs_error"]
print("Smaller absolute error with the forest:", (gain > 0).sum())
print("Larger absolute error with the forest: ", (gain < 0).sum())

## 実習　一行の予測と全体の誤差を伝える

### 1. 検証行を一つ選ぶ（約 5 分）

下の表から，調べたい `record_id` を一つ選び，材齢や材料の量など，どこに注目したかを書きます。別の検証行を選ぶ場合は，セルを足して `validation` の表を表示してもかまいません。

図を作る前に，木の予測がすべて実測を下回る場合を考えます。平均すると実測に届くでしょうか。実測より低い予測と高い予測が両方ある場合にはどうでしょうか。理由も書きます。

（選んだ記録，理由，平均についての考えを記入）

In [ ]:
candidate_ids = [5, 56, 105, 131]
validation.loc[candidate_ids, ["recipe_id", "age", "cement", "water"]]

### 2. 木の予測と平均を読む（約 9 分）

`record_id` の `None` を選んだ番号に替えて，表と図を作ります。木の予測は実測の両側にありますか。それとも片側に偏っていますか。森の平均にどちら向きのずれが残っているかを見ます。深さ制限なしの一本の木と森のどちらが実測に近いかは，表の絶対誤差で確かめます。

In [ ]:
record_id = None

if record_id not in evaluation.index:
    print("検証の表にある record_id を一つ指定します。")
else:
    one_row = X_valid.loc[[record_id]]
    student_predictions = predict_from_each_tree(forest, one_row)
    selected_row = evaluation.loc[record_id]
    display(evaluation.loc[[record_id]].round(2))

    fig, ax = plt.subplots(figsize=(7.2, 3.5), layout="constrained")
    ax.hist(student_predictions, bins=12, color="#9abcc0", edgecolor="white")
    ax.axvline(selected_row["strength"], color="#283845", linewidth=2, label="Observed")
    ax.axvline(selected_row["single_prediction"], color="#596878", linewidth=1.6,
               linestyle="--", label="Single tree")
    ax.axvline(selected_row["forest_prediction"], color="#a34f30", linewidth=2,
               linestyle="-.", label="Forest mean")
    ax.set(title=f"Record {record_id}", xlabel="Compressive strength (MPa)",
           ylabel="Number of trees")
    ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.15),
              ncols=3, fontsize=9)
    plt.show()

### 3. 一行と全体の結果を報告する（約 6 分）

強度を見積もる人に向けて，3〜4 文で報告します。

1. 選んだ記録の番号・材齢と実測を示す。木の予測が実測の両側にあるか，片側に偏るかを根拠に，森の平均に残ったずれを説明する。
2. 同じ行の深さ制限なしの一本と森を比べる。検証 208 行の全体ではどうだったかを，MAE の実数とともに書く。
3. どのような配合や材齢の予測にも使いたいかを考え，次はどの記録を集め，何を比べたいかを書く。

（ここに報告を記入）

カーネルを再起動し，上から全セルを実行します。選んだ行の表・図と報告を残したノートブックを提出します。

### データの出典

Copyright Prof. I-Cheng Yeh. Yeh, I. (1998). *Concrete Compressive Strength*. UCI Machine Learning Repository. 原表を CSV に変換し，列名，原表の行番号，配合の識別番号と授業用の分割を整えました。 [データ配布元](https://doi.org/10.24432/C5PK67) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) ・ [原著論文](https://doi.org/10.1016/S0008-8846(98)00165-3)